# Эксперименты с состязательными атаками



## 1. Установка зависимостей

In [ ]:
# !pip install -q -r ../requirements.txt
# !pip install -q spacy && python -m spacy download en_core_web_sm

## 2. Импорты

In [ ]:
import sys, random, re, inspect
import numpy as np
import pandas as pd
from pathlib import Path

sys.path.insert(0, str(Path("..").resolve()))

from detector import radar_score
from metrics import perplexity, human_readability_score
from rules import RULES
from framework import TextBreakFramework, build_rulebook, add_broken_columns

## 3. Проверка детектора RADAR

In [ ]:
text_ai = (
    "Artificial intelligence has rapidly evolved over the past decade, "
    "fundamentally reshaping a wide range of industries. Recent advances "
    "in transformer-based architectures have enabled models to learn rich "
    "contextual representations at scale."
)
text_human = (
    "With all due respect, you are a hypocrite. You accuse others of bullying "
    "yet you repeatedly bully people for various things, including barrating me "
    "for my spelling despite me telling you that i am dyslexic."
)

for label, text in [("AI", text_ai), ("Human", text_human)]:
    p_ai, p_h, _ = radar_score(text)
    print(f"{label:6s} → prob_ai={p_ai:.3f}, prob_human={p_h:.3f}")

## 4. Вспомогательные метрики

In [ ]:
ppl = perplexity(text_ai)
read_score, details = human_readability_score(text_ai)
print(f"Перплексия:  {ppl:.1f}")
print(f"Читаемость:  {read_score:.3f}")
print("Компоненты:", details["components"])

## 5. Демонстрация всех правил

In [ ]:
from src.rules import (
    remove_punctuation, add_random_punct,
    smart_delete_words, swap_phrases_spacy, add_stopwords,
    homoglypgs_random, swap_chars_random, insert_random_char,
    double_random_letter, delete_random_letter,
    keyboard_miss, replace_equivalents,
    roberta_synonyms, grammar_distort,
)

TEXT = text_ai
for name, fn, kwargs in [
    ("remove_punctuation",  remove_punctuation,  {}),
    ("add_random_punct",     add_random_punct,     {"p": 0.3, "seed": 42}),
    ("smart_delete_words",   smart_delete_words,   {"p": 0.3, "seed": 42}),
    ("add_stopwords",        add_stopwords,        {"p": 0.5, "seed": 42}),
    ("swap_chars_random",    swap_chars_random,    {"p": 0.1, "seed": 42}),
    ("insert_random_char",   insert_random_char,   {"p": 0.05,"seed": 42}),
    ("double_random_letter", double_random_letter, {"p": 0.2, "seed": 42}),
    ("delete_random_letter", delete_random_letter, {"p": 0.2, "seed": 42}),
    ("grammar_distort",      grammar_distort,      {"p": 0.3, "seed": 42}),
]:
    out = fn(TEXT, **kwargs)
    s0  = radar_score(TEXT)[0]
    s1  = radar_score(out)[0]
    print(f"{name:25s}  {s0:.3f} -> {s1:.3f}")

## 6. Фреймворк

In [ ]:
rulebook = build_rulebook(RULES)

fw = TextBreakFramework(
    rulebook=rulebook,
    radar_score_fn=radar_score,
    perplexity_fn=perplexity,
    readability_fn=human_readability_score,
    min_readability=0.90,
    max_ppl_ratio=6.0,
)

print("Правила:", list(rulebook.keys()))

## 7. Лучшее правило для одного текста

In [ ]:
df = fw.best_per_rule(TEXT, n_runs=5, strength=0.5, seed=42)
df[["rule", "base_radar_ai", "radar_ai", "ppl", "read"]]

## 8. Проверка правил на MAGE (машинные тексты)

In [ ]:
from datasets import concatenate_datasets

dataset_mage = load_dataset("yaful/MAGE")

machine_mage = dataset_mage["train"].filter(lambda x: "machine" in x["src"]).shuffle(seed=42).select(range(1000))
human_mage   = dataset_mage["train"].filter(lambda x: "human"   in x["src"]).shuffle(seed=42).select(range(1000))

def _prep_mage(split, min_chars=300, max_chars=1200):
    texts = []
    for ex in split:
        t = ex["text"].strip()
        if len(t) < min_chars: continue
        texts.append(t[:max_chars])
    return texts

machine_texts = _prep_mage(machine_mage)
human_texts   = _prep_mage(human_mage)
print(f"MAGE — машинных: {len(machine_texts)}, человеческих: {len(human_texts)}")

In [ ]:
DATASET_MACHINE = machine_texts[:50]

results_machine = []
base_s = {i: radar_score(t)[0] for i, t in enumerate(DATASET_MACHINE)}
base_p = {i: perplexity(t)      for i, t in enumerate(DATASET_MACHINE)}
base_r = {i: human_readability_score(t)[0] for i, t in enumerate(DATASET_MACHINE)}

for rule_name, rule_fn in RULES.items():
    sig = inspect.signature(rule_fn)
    for i, text in enumerate(DATASET_MACHINE):
        for r in range(N_RUNS):
            seed = SEED_BASE + r
            kw = {}
            if "seed" in sig.parameters: kw["seed"] = seed
            elif "rng" in sig.parameters: kw["rng"] = random.Random(seed)
            else: random.seed(seed)
            out = rule_fn(text, **kw)
            results_machine.append({
                "rule": rule_name, "text_id": i, "run": r,
                "radar_drop": base_s[i] - radar_score(out)[0],
                "ppl_before": base_p[i], "ppl_after": perplexity(out),
                "read_before": base_r[i], "read_after": human_readability_score(out)[0],
            })

df_m = pd.DataFrame(results_machine)
top_m = df_m.groupby(["rule","text_id"])["radar_drop"].mean().groupby("rule").mean().sort_values(ascending=False)
print("Снижение radar_ai на машинных текстах MAGE:")
print(top_m.round(4))

## 9. Фреймворк: поиск лучшей комбинации правил

In [ ]:
for i, text in enumerate(machine_texts[:10]):
    result = fw.generate_best_candidate(text, tries=10, strength=0.5, seed=100+i)
    s0 = result["base"]["radar_ai"]
    s1 = result["cand"]["radar_ai"]
    applied = [x["rule"] for x in result["log"]]
    print(f"[{i}] {s0:.3f} -> {s1:.3f}  правила: {applied}")

## 10. Подготовка многоуровневого датасета для дообучения

In [ ]:
def make_split_mage(split, N=None, seed=42, min_chars=300, max_chars=1200):
    def _clip(ex):
        t = (ex["text"] or "").strip()
        return {"ok": len(t) >= min_chars, "text": t[:max_chars]}
    def _prep(ds):
        if "label" in ds.column_names:
            ds = ds.rename_column("label", "mage_label")
        return ds.map(_clip).filter(lambda x: x["ok"]).remove_columns(["ok"])
    h = _prep(split.filter(lambda x: "human"   in x["src"]))
    m = _prep(split.filter(lambda x: "machine" in x["src"]))
    if N:
        h = h.shuffle(seed=seed).select(range(min(N, len(h))))
        m = m.shuffle(seed=seed).select(range(min(N, len(m))))
    h = h.add_column("label", [1]*len(h))
    m = m.add_column("label", [0]*len(m))
    return concatenate_datasets([h, m]).shuffle(seed=seed)

clean_train = make_split_mage(dataset_mage["train"],      N=2000, seed=42)
clean_val   = make_split_mage(dataset_mage["validation"], N=500,  seed=43)
clean_test  = make_split_mage(dataset_mage["test"],       N=500,  seed=44)
print(f"Train: {len(clean_train)}, Val: {len(clean_val)}, Test: {len(clean_test)}")

In [ ]:
# Сборка многоуровневого датасета с разными степенями искажения
# (ДОЛГО — раскомментируйте при запуске на GPU)

# from datasets import DatasetDict
#
# DISTORTION_CONFIGS = {
#     "clean":  {"enabled": False, "strength": 0.00, "tries":  0, "k_max": 0},
#     "weak":   {"enabled": True,  "strength": 0.30, "tries":  4, "k_max": 2},
#     "medium": {"enabled": True,  "strength": 0.55, "tries":  8, "k_max": 3},
#     "strong": {"enabled": True,  "strength": 0.85, "tries": 12, "k_max": 3},
# }
# level_probs = {"clean": 0.20, "weak": 0.25, "medium": 0.25, "strong": 0.30}
#
# broken_train = add_broken_columns(clean_train, fw, seed=42, tries=8, strength=0.6)
# broken_val   = add_broken_columns(clean_val,   fw, seed=43, tries=8, strength=0.6)
# broken_test  = add_broken_columns(clean_test,  fw, seed=44, tries=8, strength=0.6)
#
# DatasetDict({"train": broken_train, "validation": broken_val, "test": broken_test})\
#     .save_to_disk("../data/broken_multilevel_dataset")
print("Датасет сохраняется командами выше. Раскомментируйте для запуска.")

## 11. Дообучение RADAR

Полный пайплайн дообучения RADAR-Vicuna-7B: дообучение только на чистых, смешанных, только атакованных данных.

In [ ]:
import os, gc, random, json
import numpy as np
import pandas as pd
import torch
from datasets import load_from_disk, concatenate_datasets, Dataset, DatasetDict
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, DataCollatorWithPadding,
)
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
    precision_recall_fscore_support,
)

BASE_MODEL_NAME  = "TrustSafeAI/RADAR-Vicuna-7B"
DATASET_PATH     = "data/broken_multilevel_dataset"
CLEAN_MODEL_DIR  = "models/radar_clean_only_finetuned"
MIXED_MODEL_DIR  = "models/radar_mixed_finetuned"
MAX_LENGTH  = 512
SEED        = 42
LABEL_COL   = "label"
TRAIN_TEXT_COL = "train_text"
USE_FP16    = torch.cuda.is_available()
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"
id2label    = {0: "machine", 1: "human"}
label2id    = {"machine": 0, "human": 1}

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

set_seed(SEED)
os.makedirs(CLEAN_MODEL_DIR, exist_ok=True)
os.makedirs(MIXED_MODEL_DIR, exist_ok=True)
os.makedirs("runs", exist_ok=True)
print("Config OK. Device:", DEVICE)

In [ ]:
# Загрузка датасета
dataset = load_from_disk(DATASET_PATH)
train_ds     = dataset["train"]
val_ds       = dataset["validation"]
test_clean_ds = dataset["test_clean_eval"]
test_aug_ds   = dataset["test_aug_eval"]

print(dataset)
print("Train labels:", dict(zip(*np.unique(train_ds["label"], return_counts=True))))
print("Train levels:", dict(zip(*np.unique(train_ds["distortion_level"], return_counts=True))))

In [ ]:
# Вспомогательные функции
def add_train_text_col(ds):
    if TRAIN_TEXT_COL in ds.column_names:
        return ds
    def _map(ex):
        level = ex.get("distortion_level", "clean")
        return {TRAIN_TEXT_COL: ex["text"] if level == "clean" else ex.get("broken_text", ex["text"])}
    return ds.map(_map)

def ensure_col(ds, col, src_col):
    if col in ds.column_names: return ds
    return ds.map(lambda ex: {col: ex[src_col]})

def tokenize_ds(ds, tokenizer, text_col, label_col="label", max_length=512):
    tok = ds.map(lambda b: tokenizer(b[text_col], truncation=True, max_length=max_length), batched=True)
    keep = [c for c in ["input_ids","attention_mask",label_col] if c in tok.column_names]
    tok = tok.select_columns(keep)
    if label_col != "labels": tok = tok.rename_column(label_col, "labels")
    return tok

def compute_metrics_machine_positive(eval_pred):
    logits, labels = eval_pred
    probs = np.exp(logits - logits.max(1, keepdims=True))
    probs /= probs.sum(1, keepdims=True)
    machine_probs  = probs[:, 0]
    machine_labels = 1 - labels
    machine_preds  = (logits.argmax(1) == 0).astype(int)
    out = {
        "accuracy":  accuracy_score(machine_labels, machine_preds),
        "f1":        f1_score(machine_labels, machine_preds, zero_division=0),
        "precision": precision_score(machine_labels, machine_preds, zero_division=0),
        "recall":    recall_score(machine_labels, machine_preds, zero_division=0),
    }
    try: out["roc_auc"] = roc_auc_score(machine_labels, machine_probs)
    except: out["roc_auc"] = float("nan")
    return out

def get_probs_and_labels(trainer, ds):
    pred   = trainer.predict(ds)
    logits = pred.predictions; labels = pred.label_ids
    probs  = np.exp(logits - logits.max(1, keepdims=True))
    probs /= probs.sum(1, keepdims=True)
    return probs, labels

def convert_to_machine_positive(labels, probs):
    return 1 - labels, probs[:, 0]

def find_best_threshold(labels, pos_probs):
    best = None; best_f1 = -1.0
    for thr in np.arange(0.05, 0.96, 0.01):
        preds = (pos_probs >= thr).astype(int)
        f1 = f1_score(labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best = {"threshold": float(thr), "f1": float(f1),
                    "precision": float(precision_score(labels, preds, zero_division=0)),
                    "recall":    float(recall_score(labels, preds, zero_division=0)),
                    "roc_auc":   float(roc_auc_score(labels, pos_probs))}
    return best

def metrics_from_probs(labels, pos_probs, threshold):
    preds = (pos_probs >= threshold).astype(int)
    return {"accuracy":  float(accuracy_score(labels, preds)),
            "f1":        float(f1_score(labels, preds, zero_division=0)),
            "precision": float(precision_score(labels, preds, zero_division=0)),
            "recall":    float(recall_score(labels, preds, zero_division=0)),
            "roc_auc":   float(roc_auc_score(labels, pos_probs))}

def build_eval_trainer(model_path, tokenizer):
    eval_model = AutoModelForSequenceClassification.from_pretrained(
        model_path, num_labels=2, id2label=id2label, label2id=label2id)
    eval_args = TrainingArguments(
        output_dir=f"runs/eval_{os.path.basename(model_path.rstrip('/'))}",
        per_device_eval_batch_size=16, report_to="none", fp16=USE_FP16)
    return Trainer(model=eval_model, args=eval_args,
                   processing_class=tokenizer,
                   data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
                   compute_metrics=compute_metrics_machine_positive)

print("Functions defined.")

### 11a. Дообучение RADAR на чистых данных

In [ ]:
train_clean = dataset["train"].filter(lambda x: x["distortion_level"] == "clean")
val_clean   = dataset["validation_clean"] if "validation_clean" in dataset else dataset["validation"].filter(lambda x: x["distortion_level"] == "clean")

print("Clean train:", len(train_clean))
print("Clean val:  ", len(val_clean))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

train_tok_clean = tokenize_ds(train_clean, tokenizer, "text", LABEL_COL, MAX_LENGTH)
val_tok_clean   = tokenize_ds(val_clean,   tokenizer, "text", LABEL_COL, MAX_LENGTH)

model_clean = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

training_args_clean = TrainingArguments(
    output_dir=CLEAN_MODEL_DIR,
    eval_strategy="epoch", save_strategy="epoch",
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    num_train_epochs=3, learning_rate=1e-5, weight_decay=0.01,
    warmup_ratio=0.1, load_best_model_at_end=True,
    metric_for_best_model="roc_auc", greater_is_better=True,
    save_total_limit=2, report_to="none", fp16=USE_FP16, seed=SEED,
)
trainer_clean = Trainer(
    model=model_clean, args=training_args_clean,
    train_dataset=train_tok_clean, eval_dataset=val_tok_clean,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics_machine_positive,
)
trainer_clean.train()
trainer_clean.save_model(CLEAN_MODEL_DIR)
tokenizer.save_pretrained(CLEAN_MODEL_DIR)
print("Clean-only model saved to:", CLEAN_MODEL_DIR)

### 11b. Дообучение RADAR на смешанном датасете

In [ ]:
train_mixed = add_train_text_col(dataset["train"])
val_mixed   = add_train_text_col(dataset["validation"])

test_clean_eval = ensure_col(dataset["test_clean_eval"], "input_text", "text")
test_aug_eval   = ensure_col(dataset["test_aug_eval"],   "input_text", "broken_text")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)

train_tok_mixed = tokenize_ds(train_mixed, tokenizer, TRAIN_TEXT_COL, LABEL_COL, MAX_LENGTH)
val_tok_mixed   = tokenize_ds(val_mixed,   tokenizer, TRAIN_TEXT_COL, LABEL_COL, MAX_LENGTH)

model_mixed = AutoModelForSequenceClassification.from_pretrained(
    BASE_MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

training_args_mixed = TrainingArguments(
    output_dir=MIXED_MODEL_DIR,
    eval_strategy="epoch", save_strategy="epoch",
    per_device_train_batch_size=8, per_device_eval_batch_size=16,
    num_train_epochs=3, learning_rate=1e-5, weight_decay=0.01,
    warmup_ratio=0.1, load_best_model_at_end=True,
    metric_for_best_model="roc_auc", greater_is_better=True,
    save_total_limit=2, report_to="none", fp16=USE_FP16, seed=SEED,
)
trainer_mixed = Trainer(
    model=model_mixed, args=training_args_mixed,
    train_dataset=train_tok_mixed, eval_dataset=val_tok_mixed,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer=tokenizer),
    compute_metrics=compute_metrics_machine_positive,
)
trainer_mixed.train()
trainer_mixed.save_model(MIXED_MODEL_DIR)
tokenizer.save_pretrained(MIXED_MODEL_DIR)
print("Mixed model saved to:", MIXED_MODEL_DIR)

### 12c. Оценка: базовый детектор vs дообучение на чистых vs дообучение на смешанных

In [ ]:
def evaluate_model_on_split(model_name, model_path, eval_ds, val_ds, text_col, split_name):
    local_tok = AutoTokenizer.from_pretrained(model_path, use_fast=True)
    eval_trainer = build_eval_trainer(model_path, local_tok)

    val_col  = TRAIN_TEXT_COL if TRAIN_TEXT_COL in val_ds.column_names else "input_text"
    tok_val  = tokenize_ds(val_ds,  local_tok, val_col,  LABEL_COL, MAX_LENGTH)
    tok_test = tokenize_ds(eval_ds, local_tok, text_col, LABEL_COL, MAX_LENGTH)

    val_probs,  val_labels  = get_probs_and_labels(eval_trainer, tok_val)
    val_ml, val_mp = convert_to_machine_positive(val_labels, val_probs)
    best_thr = find_best_threshold(val_ml, val_mp)

    test_probs, test_labels = get_probs_and_labels(eval_trainer, tok_test)
    test_ml, test_mp = convert_to_machine_positive(test_labels, test_probs)

    argmax_m = metrics_from_probs(test_ml, test_mp, 0.5)
    thr_m    = metrics_from_probs(test_ml, test_mp, best_thr["threshold"])

    rows = [
        {"model": model_name, "eval_split": split_name, "mode": "argmax",
         "threshold": 0.5, **argmax_m,
         "val_best_threshold": best_thr["threshold"], "val_f1": best_thr["f1"]},
        {"model": model_name, "eval_split": split_name, "mode": "best_threshold",
         "threshold": best_thr["threshold"], **thr_m,
         "val_best_threshold": best_thr["threshold"], "val_f1": best_thr["f1"]},
    ]
    return pd.DataFrame(rows), best_thr

MODELS = {
    "base_radar":              BASE_MODEL_NAME,
    "clean_only_finetuned":    CLEAN_MODEL_DIR,
    "mixed_finetuned":         MIXED_MODEL_DIR,
}
EVAL_SPLITS = {
    "test_clean_eval": (test_clean_eval, "input_text"),
    "test_aug_eval":   (test_aug_eval,   "input_text"),
}

all_res = []
for mname, mpath in MODELS.items():
    for sname, (sds, tcol) in EVAL_SPLITS.items():
        print(f"Eval: {mname} / {sname}")
        df, _ = evaluate_model_on_split(mname, mpath, sds, val_mixed, tcol, sname)
        all_res.append(df)

results_df = pd.concat(all_res, ignore_index=True)
main = results_df[results_df["mode"] == "best_threshold"].copy()

print("\n=== F1 ===")
print(main.pivot_table(index="eval_split", columns="model", values="f1").round(3))
print("\n=== ROC-AUC ===")
print(main.pivot_table(index="eval_split", columns="model", values="roc_auc").round(3))
main[["model","eval_split","f1","precision","recall","roc_auc","threshold"]]    .sort_values(["eval_split","f1"], ascending=[True,False])

### 11d. Состязательное обучение

In [ ]:
# Подготовка датасетов
def prepare_adv_split(dataset_split):
    df = dataset_split.to_pandas().copy()
    df["label"] = 1 - df["label"]
    clean_df = df[["text","label"]].copy(); clean_df["source"] = "clean"

    if "broken_text" in df.columns:
        attacked_text = np.where(df["label"]==1,
                                 df["broken_text"].fillna(df["text"]),
                                 df["text"])
    else:
        attacked_text = df["text"]

    attacked_df = pd.DataFrame({"text": attacked_text,
                                 "label": df["label"].values, "source": "attacked"})
    return clean_df, attacked_df

def balance_df_adv(df, seed=42):
    min_n = df["label"].value_counts().min()
    return (df.groupby("label", group_keys=False)
              .apply(lambda x: x.sample(min_n, random_state=seed))
              .sample(frac=1, random_state=seed)
              .reset_index(drop=True))

train_clean_adv, train_attacked_adv = prepare_adv_split(dataset["train"])
val_clean_adv,   val_attacked_adv   = prepare_adv_split(dataset["validation"])
test_clean_adv,  test_attacked_adv  = prepare_adv_split(
    dataset.get("test_aug", dataset.get("test_aug_eval", dataset["test"])))

train_clean_b  = balance_df_adv(train_clean_adv)
train_attack_b = balance_df_adv(train_attacked_adv)
train_mixed_b  = balance_df_adv(pd.concat([train_clean_adv, train_attacked_adv]))
val_clean_b    = balance_df_adv(val_clean_adv)
val_attack_b   = balance_df_adv(val_attacked_adv)
val_mixed_b    = balance_df_adv(pd.concat([val_clean_adv, val_attacked_adv]))

print("clean train:", train_clean_b.shape)
print("attacked train:", train_attack_b.shape)
print("mixed train:", train_mixed_b.shape)

In [ ]:
# Функция обучения RADAR детектора
def train_radar_detector_adv(model_name, train_df, val_df, output_dir,
                              max_length=512, epochs=2, batch_size=8, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True, trust_remote_code=True)

    hf = DatasetDict({
        "train": Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True)),
        "val":   Dataset.from_pandas(val_df[["text","label"]].reset_index(drop=True)),
    })
    hf = hf.map(lambda b: tok(b["text"], truncation=True, max_length=max_length), batched=True)
    hf = hf.rename_column("label","labels"); hf.set_format("torch")

    def _metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
        return compute_metrics_from_probs(labels, probs)

    args = TrainingArguments(
        output_dir=output_dir, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, num_train_epochs=epochs,
        weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="eval_f1", greater_is_better=True,
        logging_steps=50, save_total_limit=1, report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=hf["train"], eval_dataset=hf["val"],
                      processing_class=tok,
                      data_collator=DataCollatorWithPadding(tokenizer=tok),
                      compute_metrics=_metrics)
    trainer.train()
    trainer.save_model(output_dir); tok.save_pretrained(output_dir)
    return output_dir

ADV_OUT_DIR = "/content/radar_adversarial_training"
os.makedirs(ADV_OUT_DIR, exist_ok=True)

experiments_adv = {
    "clean_only":     (train_clean_b,  val_clean_b),
    "attacked_only":  (train_attack_b, val_attack_b),
    "mixed":          (train_mixed_b,  val_mixed_b),
}
trained_paths_adv = {}
for name, (tr, vl) in experiments_adv.items():
    print(f"\nTraining: {name}")
    path = train_radar_detector_adv(
        BASE_MODEL_NAME, tr, vl,
        output_dir=os.path.join(ADV_OUT_DIR, name),
    )
    trained_paths_adv[name] = path
    gc.collect(); torch.cuda.empty_cache()

print("Done:", trained_paths_adv)

In [ ]:
# Оценка всех моделей на чистых и атакованных тестах
def eval_adv_model(model_path, val_df, test_clean_df, test_attacked_df, model_name):
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, trust_remote_code=True).to(DEVICE)
    model.eval()

    def predict(df, max_length=512, batch_size=16):
        scores = []
        for i in range(0, len(df), batch_size):
            texts = df["text"].iloc[i:i+batch_size].tolist()
            enc = tok(texts, truncation=True, padding=True,
                      max_length=max_length, return_tensors="pt").to(DEVICE)
            with torch.no_grad():
                p = torch.softmax(model(**enc).logits, dim=-1)[:,1].cpu().numpy()
            scores.extend(p.tolist())
        return np.array(scores)

    val_probs = predict(val_df)
    val_labels = val_df["label"].values
    best_t, _ = find_best_threshold(val_labels, val_probs)

    rows = []
    for split_name, df in [("test_clean", test_clean_df), ("test_attacked", test_attacked_df)]:
        probs  = predict(df)
        labels = df["label"].values
        m = compute_metrics_from_probs(labels, probs, threshold=best_t)
        rows.append({"model": model_name, "split": split_name, "threshold": best_t, **m})

    del model; gc.collect(); torch.cuda.empty_cache()
    return rows

eval_rows_adv = []
for exp_name, model_path in trained_paths_adv.items():
    val_for_thr = {"clean_only": val_clean_b,
                   "attacked_only": val_attack_b,
                   "mixed": val_mixed_b}[exp_name]
    rows = eval_adv_model(model_path, val_for_thr, test_clean_adv, test_attacked_adv, exp_name)
    eval_rows_adv.extend(rows)

adv_results = pd.DataFrame(eval_rows_adv)

clean_r    = adv_results[adv_results["split"]=="test_clean"].set_index("model")
attacked_r = adv_results[adv_results["split"]=="test_attacked"].set_index("model")

drop_rows = []
for m in clean_r.index:
    drop_rows.append({"model":m,
        "clean_f1":    clean_r.loc[m,"f1"],
        "attacked_f1": attacked_r.loc[m,"f1"],
        "f1_drop":     clean_r.loc[m,"f1"] - attacked_r.loc[m,"f1"],
        "clean_auc":   clean_r.loc[m,"auc"],
        "attacked_auc":attacked_r.loc[m,"auc"],
        "auc_drop":    clean_r.loc[m,"auc"] - attacked_r.loc[m,"auc"],
    })

adv_drop_df = pd.DataFrame(drop_rows)
adv_drop_df.to_csv(os.path.join(ADV_OUT_DIR, "drop_summary.csv"), index=False)
print(adv_drop_df[["model","clean_f1","attacked_f1","f1_drop"]].to_string(index=False))

### 11e.Проверка обобщающей способности

Проверяем, обобщается ли устойчивость детектора на ранее неизвестные атаки.
- Seen rules — правила, использованные при adversarial training
- Unseen rules — правила, исключённые из обучения


In [ ]:
SEEN_RULES = [
    "delete_random_letter",
    "keyboard_miss",
    "insert_random_char",
]

UNSEEN_RULES = [
    "homoglypgs_random",
    "replace_equivalents",
    "swap_chars_random",
    "double_random_letter",
]

In [ ]:
from framework import TextBreakFramework

def make_sub_fw(rule_names):
    sub_rulebook = {name: fw.rulebook[name]
                    for name in rule_names if name in fw.rulebook}
    return TextBreakFramework(
        rulebook=sub_rulebook,
        radar_score_fn=fw.radar_score,
        perplexity_fn=fw.perplexity,
        readability_fn=fw.readability,
        normalize_fn=fw.normalize,
        min_readability=fw.min_readability,
        max_ppl_ratio=fw.max_ppl_ratio,
        max_ppl_abs=fw.max_ppl_abs,
    )

seen_fw   = make_sub_fw(SEEN_RULES)
unseen_fw = make_sub_fw(UNSEEN_RULES)

print("seen_fw rules:  ", list(seen_fw.rulebook.keys()))
print("unseen_fw rules:", list(unseen_fw.rulebook.keys()))

In [ ]:
HELDOUT_DATASET_PATH = "data/broken_multilevel_dataset"
HELDOUT_SEED = 42

heldout_dataset = load_from_disk(HELDOUT_DATASET_PATH)

def load_clean_df(split):
    df = heldout_dataset[split].to_pandas()[["text", "label"]].copy()
    df["label"] = 1 - df["label"]
    return df.reset_index(drop=True)

train_clean_ho = load_clean_df("train")
val_clean_ho   = load_clean_df("validation")
test_clean_ho  = load_clean_df("test_clean" if "test_clean" in heldout_dataset else "test")

print("train:", train_clean_ho["label"].value_counts().to_dict())
print("val:  ", val_clean_ho["label"].value_counts().to_dict())
print("test: ", test_clean_ho["label"].value_counts().to_dict())

In [ ]:
def make_attacked_df(clean_df, attack_fw, seed=42, tries=8, strength=0.5, k_max=3):
    rows = []
    for i, row in tqdm(clean_df.iterrows(), total=len(clean_df)):
        text  = row["text"]
        label = int(row["label"])

        if label == 0:
            # human — не трогаем
            attacked_text = text
            rules = []
        else:
            res = attack_fw.generate_best_candidate(
                text=text, tries=tries, strength=strength,
                seed=seed + i, k_max=k_max, target_drop=0.0,
            )
            attacked_text = res["text"]
            rules = [x["rule"] for x in res["log"]]

        rows.append({"text": attacked_text, "label": label, "rules": str(rules)})

    return pd.DataFrame(rows)

print("Генерация train seen-атак...")
train_seen_attacked = make_attacked_df(
    train_clean_ho, seen_fw, seed=HELDOUT_SEED, tries=8, strength=0.5, k_max=3)

print("Генерация val seen-атак...")
val_seen_attacked = make_attacked_df(
    val_clean_ho, seen_fw, seed=HELDOUT_SEED + 10_000, tries=8, strength=0.5, k_max=3)

print("Генерация test unseen-атак...")
test_unseen_attacked = make_attacked_df(
    test_clean_ho, unseen_fw, seed=HELDOUT_SEED + 20_000, tries=8, strength=0.5, k_max=3)

print("Генерация test seen-атак...")
test_seen_attacked = make_attacked_df(
    test_clean_ho, seen_fw, seed=HELDOUT_SEED + 30_000, tries=8, strength=0.5, k_max=3)

train_texts = set(train_clean_ho["text"].astype(str))
test_texts  = set(test_clean_ho["text"].astype(str))
print("\nПересечение train/test clean:", len(train_texts & test_texts))

train_att = set(train_seen_attacked["text"].astype(str))
test_att  = set(test_unseen_attacked["text"].astype(str))
print("Пересечение train_seen / test_unseen:", len(train_att & test_att))

train_seen_attacked.to_csv("/content/train_seen_attacked.csv", index=False)
val_seen_attacked.to_csv("/content/val_seen_attacked.csv", index=False)
test_unseen_attacked.to_csv("/content/test_unseen_attacked.csv", index=False)
test_seen_attacked.to_csv("/content/test_seen_attacked.csv", index=False)
print("\nДатасеты сохранены.")

In [ ]:
def balance_df(df, seed=42):
    min_n = df["label"].value_counts().min()
    return (df.groupby("label", group_keys=False)
              .apply(lambda x: x.sample(min_n, random_state=seed))
              .sample(frac=1, random_state=seed)
              .reset_index(drop=True))

train_clean_bal = balance_df(train_clean_ho, HELDOUT_SEED)
val_clean_bal   = balance_df(val_clean_ho,   HELDOUT_SEED)

train_seen_mixed = pd.concat(
    [train_clean_ho[["text","label"]], train_seen_attacked[["text","label"]]],
    ignore_index=True,
)
val_seen_mixed = pd.concat(
    [val_clean_ho[["text","label"]], val_seen_attacked[["text","label"]]],
    ignore_index=True,
)
train_seen_mixed_bal = balance_df(train_seen_mixed, HELDOUT_SEED)
val_seen_mixed_bal   = balance_df(val_seen_mixed,   HELDOUT_SEED)

print("clean-only train size:    ", len(train_clean_bal))
print("seen-mixed train size:    ", len(train_seen_mixed_bal))
print("Label distribution (mixed):", train_seen_mixed_bal["label"].value_counts().to_dict())

In [ ]:
# Обучение двух детекторов
HELDOUT_BASE_MODEL = "TrustSafeAI/RADAR-Vicuna-7B"
HELDOUT_OUT_DIR    = "/content/radar_heldout_generalization"
HELDOUT_MAX_LEN    = 256
os.makedirs(HELDOUT_OUT_DIR, exist_ok=True)

def train_heldout_detector(model_name, train_df, val_df, output_dir,
                            max_length=256, epochs=2, batch_size=2,
                            lr=2e-5, grad_accum=4):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True, trust_remote_code=True)

    hf = DatasetDict({
        "train": Dataset.from_pandas(train_df[["text","label"]].reset_index(drop=True)),
        "val":   Dataset.from_pandas(val_df[["text","label"]].reset_index(drop=True)),
    })
    hf = hf.map(lambda b: tok(b["text"], truncation=True, max_length=max_length), batched=True)
    hf = hf.rename_column("label","labels"); hf.set_format("torch")

    args = TrainingArguments(
        output_dir=output_dir, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size,
        gradient_accumulation_steps=grad_accum,
        num_train_epochs=epochs, weight_decay=0.01,
        load_best_model_at_end=True, metric_for_best_model="eval_f1",
        greater_is_better=True, logging_steps=50, save_total_limit=1,
        report_to="none", fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=hf["train"], eval_dataset=hf["val"],
                      processing_class=tok,
                      data_collator=DataCollatorWithPadding(tokenizer=tok),
                      compute_metrics=trainer_metrics)
    trainer.train()
    trainer.save_model(output_dir); tok.save_pretrained(output_dir)
    del trainer; gc.collect(); torch.cuda.empty_cache()
    return output_dir

heldout_train_configs = {
    "radar_clean_only":        (train_clean_bal,      val_clean_bal),
    "radar_seen_attacks_mixed":(train_seen_mixed_bal, val_seen_mixed_bal),
}

heldout_paths = {}
for exp_name, (tr, vl) in heldout_train_configs.items():
    print(f"\nTraining: {exp_name}  (train={len(tr)}, val={len(vl)})")
    path = train_heldout_detector(
        model_name=HELDOUT_BASE_MODEL, train_df=tr, val_df=vl,
        output_dir=os.path.join(HELDOUT_OUT_DIR, exp_name),
        max_length=HELDOUT_MAX_LEN, epochs=2, batch_size=2, grad_accum=4,
    )
    heldout_paths[exp_name] = path

print("\nСохранённые модели:", heldout_paths)

In [ ]:
def predict_probs(df, model_path, max_length=256, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_path, use_fast=True, trust_remote_code=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_path, trust_remote_code=True).to(DEVICE)
    model.eval()
    scores = []
    texts = df["text"].tolist()
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc = tok(batch, truncation=True, padding=True,
                  max_length=max_length, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            p = torch.softmax(model(**enc).logits, dim=-1)[:, 1].cpu().numpy()
        scores.extend(p.tolist())
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return np.array(scores)

eval_splits = {
    "test_clean":           test_clean_ho,
    "test_seen_attacked":   test_seen_attacked,
    "test_unseen_attacked": test_unseen_attacked,
}

heldout_rows = []

for model_name, model_path in heldout_paths.items():
    print(f"\nEvaluating: {model_name}")

    val_df = val_clean_bal if model_name == "radar_clean_only" else val_seen_mixed_bal
    val_probs = predict_probs(val_df, model_path, max_length=HELDOUT_MAX_LEN)
    threshold, _ = find_best_threshold(val_df["label"].values, val_probs)
    print(f"  Best threshold (val): {threshold:.2f}")

    for split_name, df in eval_splits.items():
        probs  = predict_probs(df, model_path, max_length=HELDOUT_MAX_LEN)
        labels = df["label"].values
        m      = compute_metrics_from_probs(labels, probs, threshold=threshold)
        ai_mask = labels == 1
        heldout_rows.append({
            "model":      model_name,
            "split":      split_name,
            "threshold":  threshold,
            **m,
            "machine_score_mean": float(np.mean(probs[ai_mask])),
        })
        print(f"  {split_name}: F1={m['f1']:.3f}, AUC={m['auc']:.3f}")

heldout_df = pd.DataFrame(heldout_rows)
heldout_df.to_csv("/content/heldout_generalization_eval.csv", index=False)
heldout_df

In [ ]:
def make_drop_table(eval_df):
    clean = eval_df[eval_df["split"] == "test_clean"].set_index("model")
    rows  = []
    for split in ["test_seen_attacked", "test_unseen_attacked"]:
        attacked = eval_df[eval_df["split"] == split].set_index("model")
        for model_name in clean.index:
            rows.append({
                "model":           model_name,
                "attack_split":    split,
                "clean_f1":        round(clean.loc[model_name, "f1"],   3),
                "attacked_f1":     round(attacked.loc[model_name, "f1"], 3),
                "f1_drop":         round(clean.loc[model_name, "f1"] - attacked.loc[model_name, "f1"], 3),
                "clean_auc":       round(clean.loc[model_name, "auc"],   3),
                "attacked_auc":    round(attacked.loc[model_name, "auc"], 3),
                "auc_drop":        round(clean.loc[model_name, "auc"] - attacked.loc[model_name, "auc"], 3),
                "machine_score_drop": round(
                    clean.loc[model_name, "machine_score_mean"]
                    - attacked.loc[model_name, "machine_score_mean"], 3),
            })
    return pd.DataFrame(rows)

drop_df = make_drop_table(heldout_df)
drop_df.to_csv("/content/heldout_generalization_drop.csv", index=False)

print(drop_df[["model","attack_split","clean_f1","attacked_f1",
               "f1_drop","auc_drop"]].to_string(index=False))

## 12. Эксперименты с размером датасета и seed


In [ ]:
LEVELS = ["clean", "weak", "medium", "strong"]
LABELS = [0, 1]
SIZE_SWEEP  = [5000, 7000, 10000, 13000, 15000]
RUN_SEEDS   = [42, 43, 44]
SWEEP_DIR   = "runs/size_seed_sweep"
os.makedirs(SWEEP_DIR, exist_ok=True)
dataset = load_from_disk(DATASET_PATH)
train_ds  = add_train_text_col(dataset["train"])
val_ds    = add_train_text_col(dataset["validation"])
test_clean_eval = ensure_col(dataset["test_clean_eval"], "input_text", "text")
test_aug_eval   = ensure_col(dataset["test_aug_eval"],   "input_text", "broken_text")

test_splits_raw = {
    "test_clean_eval": (test_clean_eval, "input_text"),
    "test_aug_eval":   (test_aug_eval,   "input_text"),
}

print("Train levels:", Counter(train_ds["distortion_level"]))
print("Train labels:", Counter(train_ds["label"]))

In [ ]:
def build_train_buckets(train_ds):
    buckets = {}
    for level in LEVELS:
        for label in LABELS:
            key = (level, label)
            buckets[key] = train_ds.filter(
                lambda x: x["distortion_level"] == level and x["label"] == label
            )
            print(f"  {key}: {len(buckets[key])}")
    return buckets

train_buckets = build_train_buckets(train_ds)

max_size = min(len(ds) for ds in train_buckets.values()) * len(train_buckets)
candidate_sizes = [s for s in SIZE_SWEEP if s <= max_size]
print("MAX_BALANCED_SIZE:", max_size)
print("Candidate sizes:  ", candidate_sizes)

In [ ]:
def sample_ds(ds, n, seed):
    if n > len(ds):
        raise ValueError(f"Requested {n}, available {len(ds)}")
    return ds.shuffle(seed=seed).select(range(n))

def build_balanced_mix(train_buckets, total_size, seed=42):
    keys = [(l, lb) for l in LEVELS for lb in LABELS]
    base_n = total_size // len(keys)
    counts = {k: base_n for k in keys}
    for k in keys[:total_size % len(keys)]: counts[k] += 1
    for k, n in counts.items():
        if n > len(train_buckets[k]):
            raise ValueError(f"Bucket {k} too small ({len(train_buckets[k])} < {n})")
    parts = [sample_ds(train_buckets[k], counts[k], seed=seed + i*10000) for i, k in enumerate(keys)]
    return concatenate_datasets(parts).shuffle(seed=seed)

def train_one(regime_name, train_raw, val_raw, seed=42, max_length=256):
    set_seed(seed)
    out_dir = os.path.join(SWEEP_DIR, regime_name)
    tok = AutoTokenizer.from_pretrained(BASE_MODEL_NAME, use_fast=True)
    train_tok = tokenize_ds(train_raw, tok, TRAIN_TEXT_COL, LABEL_COL, max_length)
    val_tok   = tokenize_ds(val_raw,   tok, TRAIN_TEXT_COL, LABEL_COL, max_length)

    model = AutoModelForSequenceClassification.from_pretrained(
        BASE_MODEL_NAME, num_labels=2, id2label=id2label, label2id=label2id)

    args = TrainingArguments(
        output_dir=out_dir, eval_strategy="epoch", save_strategy="epoch",
        per_device_train_batch_size=2, per_device_eval_batch_size=4,
        gradient_accumulation_steps=4, num_train_epochs=2,
        learning_rate=2e-5, weight_decay=0.01, warmup_ratio=0.1,
        load_best_model_at_end=True, metric_for_best_model="f1", greater_is_better=True,
        save_total_limit=1, report_to="none", fp16=USE_FP16, seed=seed, disable_tqdm=True,
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=train_tok, eval_dataset=val_tok,
                      processing_class=tok,
                      data_collator=DataCollatorWithPadding(tokenizer=tok),
                      compute_metrics=compute_metrics_machine_positive)
    trainer.remove_callback(NotebookProgressCallback)
    trainer.train()

    val_probs, val_labels = get_probs_and_labels(trainer, val_tok)
    val_ml, val_mp = convert_to_machine_positive(val_labels, val_probs)
    best_thr = find_best_threshold(val_ml, val_mp)

    rows = []
    for sname, (sds, tcol) in test_splits_raw.items():
        test_tok = tokenize_ds(sds, tok, tcol, LABEL_COL, max_length)
        test_probs, test_labels = get_probs_and_labels(trainer, test_tok)
        test_ml, test_mp = convert_to_machine_positive(test_labels, test_probs)
        m = metrics_from_probs(test_ml, test_mp, best_thr["threshold"])
        rows.append({"eval_split": sname, "regime": regime_name, **m,
                     "val_best_threshold": best_thr["threshold"], "val_f1": best_thr["f1"]})

    val_m = metrics_from_probs(val_ml, val_mp, best_thr["threshold"])
    rows.append({"eval_split": "validation", "regime": regime_name, **val_m,
                 "val_best_threshold": best_thr["threshold"], "val_f1": best_thr["f1"]})

    del model, trainer; gc.collect(); torch.cuda.empty_cache()
    return pd.DataFrame(rows)

print("Functions ready. Starting sweep...")

In [ ]:
all_runs = []

for run_seed in RUN_SEEDS:
    for total_size in candidate_sizes:
        print(f"\n{'='*60}")
        print(f"seed={run_seed}, size={total_size}")
        ds_seed = 100_000 + run_seed * 1000 + total_size
        mixed = build_balanced_mix(train_buckets, total_size, seed=ds_seed)
        regime = f"mix_s{total_size}_seed{run_seed}"
        df = train_one(regime, mixed, val_ds, seed=run_seed, max_length=256)
        df["train_size"] = total_size
        df["run_seed"]   = run_seed
        all_runs.append(df)

size_seed_df = pd.concat(all_runs, ignore_index=True)
size_seed_df.to_csv(os.path.join(SWEEP_DIR, "results.csv"), index=False)
print("Sweep done. Shape:", size_seed_df.shape)

In [ ]:
agg_df = (
    size_seed_df
    .groupby(["train_size", "eval_split"], as_index=False)
    .agg(
        f1_mean=("f1", "mean"), f1_std=("f1", "std"),
        recall_mean=("recall", "mean"),
        roc_auc_mean=("roc_auc", "mean"),
    )
    .sort_values(["eval_split", "train_size"])
)
agg_df.to_csv(os.path.join(SWEEP_DIR, "agg.csv"), index=False)
print(agg_df[agg_df["eval_split"] == "test_aug_eval"])

In [ ]:
def detect_plateau(agg_df, split="validation", metric="f1_mean", min_delta=0.003, patience=2):
    curve = (agg_df[agg_df["eval_split"] == split]
             .sort_values("train_size")[["train_size", metric]]
             .drop_duplicates("train_size").reset_index(drop=True).copy())
    curve["delta"] = curve[metric].diff()
    streak, plateau_size = 0, None
    for i in range(1, len(curve)):
        if pd.notna(curve.loc[i, "delta"]) and curve.loc[i, "delta"] < min_delta:
            streak += 1
        else:
            streak = 0
        if streak >= patience:
            plateau_size = int(curve.loc[i - patience + 1, "train_size"])
            break
    best_row = curve.loc[curve[metric].idxmax()]
    return {"plateau_size": plateau_size, "best_size": int(best_row["train_size"]),
            "best_value": float(best_row[metric])}, curve

plateau_info, curve_df = detect_plateau(agg_df, split="validation", metric="f1_mean")
print("Plateau info:", plateau_info)
print(curve_df)

## 13. IMGTB — бенчмарк детекторов

[IMGTB](https://github.com/kinit-sk/IMGTB) — готовый бенчмарк для сравнения детекторов AI-текста.
Сравниваем: статистические методы (Entropy, Loglikelihood, LogRank),
предобученные RoBERTa, наши дообученные модели, Binoculars.

In [ ]:
# Установка и клонирование IMGTB
# В Colab: раскомментируйте
# !git clone https://github.com/kinit-sk/IMGTB.git
# %cd IMGTB
# !pip install -q -r requirements.txt

# Подготовка тестовых CSV для IMGTB
# ВАЖНО: IMGTB ожидает label: 0 = human, 1 = AI (инверсия нашей разметки)
def invert_labels(df):
    df = df.copy(); df["label"] = 1 - df["label"]; return df

dataset = load_from_disk(DATASET_PATH)

clean_df = dataset["test_clean_eval"].to_pandas()[["text", "label"]]
clean_df = invert_labels(clean_df)
clean_df.to_csv("/content/clean_test.csv", index=False)

aug_df = dataset["test_aug_eval"].to_pandas()[["broken_text", "label"]]
aug_df = aug_df.rename(columns={"broken_text": "text"})
aug_df = invert_labels(aug_df)
aug_df.to_csv("/content/attacked_test.csv", index=False)

# Также подготовим DataFrame для fine-tuned энкодеров
train_df = dataset["train"].to_pandas()[["text", "label"]]
train_df["label"] = 1 - train_df["label"]

val_df = (dataset["validation_clean"] if "validation_clean" in dataset
          else dataset["validation"].filter(lambda x: x["distortion_level"]=="clean")
         ).to_pandas()[["text", "label"]]
val_df["label"] = 1 - val_df["label"]

test_clean_df = clean_df.copy()
test_aug_df = aug_df.copy()
test_aug_df["distortion_level"] = dataset["test_aug_eval"]["distortion_level"]

print("clean_test.csv:", len(clean_df), "attacked_test.csv:", len(aug_df))

In [ ]:
# Статистические методы через IMGTB CLI
print("Команды для запуска в IMGTB:")
cmds = [
    # Статистические детекторы
    "!python benchmark.py --dataset /content/clean_test.csv csv default text label 0 --methods EntropyMetric LoglikelihoodMetric LogRankMetric --base_model_name gpt2 --name clean_stat",
    "!python benchmark.py --dataset /content/attacked_test.csv csv default text label 0 --methods EntropyMetric LoglikelihoodMetric LogRankMetric --base_model_name gpt2 --name attacked_stat",
    # Предобученные RoBERTa
    "!python benchmark.py --dataset /content/clean_test.csv csv default text label 0 --methods roberta-base-openai-detector roberta-large-openai-detector --name clean_roberta",
    "!python benchmark.py --dataset /content/attacked_test.csv csv default text label 0 --methods roberta-base-openai-detector roberta-large-openai-detector --name attacked_roberta",
]
for c in cmds:
    print(c)

In [ ]:
# Fine-tuning энкодеров (RoBERTa-base и ModernBERT-base)
def compute_metrics_from_probs(y_true, probs, threshold=0.5):
    y_true = np.array(y_true); probs = np.array(probs)
    y_pred = (probs >= threshold).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average="binary", zero_division=0)
    return {"accuracy": accuracy_score(y_true, y_pred), "precision": float(precision),
            "recall": float(recall), "f1": float(f1), "auc": roc_auc_score(y_true, probs)}

def find_best_thr_encoder(y_true, probs):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.99, 99):
        f1 = compute_metrics_from_probs(y_true, probs, t)["f1"]
        if f1 > best_f1: best_f1 = f1; best_t = t
    return best_t, best_f1

def evaluate_by_distortion_level(test_aug_df, probs, threshold):
    temp = test_aug_df.copy(); temp["prob_ai"] = probs
    rows = []
    for level, group in temp.groupby("distortion_level"):
        m = compute_metrics_from_probs(group["label"].values, group["prob_ai"].values, threshold)
        m["distortion_level"] = level; m["n"] = len(group)
        rows.append(m)
    return pd.DataFrame(rows).sort_values("distortion_level")

def train_encoder(model_name, train_df, val_df, test_clean_df, test_aug_df,
                  output_dir, max_length=512, epochs=2, batch_size=8, lr=2e-5):
    tok = AutoTokenizer.from_pretrained(model_name, use_fast=True)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, ignore_mismatched_sizes=True)

    hf = DatasetDict({
        "train":      Dataset.from_pandas(train_df.reset_index(drop=True)),
        "validation": Dataset.from_pandas(val_df.reset_index(drop=True)),
        "test_clean": Dataset.from_pandas(test_clean_df.reset_index(drop=True)),
        "test_aug":   Dataset.from_pandas(test_aug_df[["text","label"]].reset_index(drop=True)),
    })
    def tokenize(batch):
        return tok(batch["text"], truncation=True, max_length=max_length)
    hf = hf.map(tokenize, batched=True).rename_column("label","labels")
    hf.set_format("torch")

    def trainer_metrics(eval_pred):
        logits, labels = eval_pred
        probs = torch.softmax(torch.tensor(logits), dim=-1)[:,1].numpy()
        return compute_metrics_from_probs(labels, probs)

    args = TrainingArguments(
        output_dir=output_dir, eval_strategy="epoch", save_strategy="epoch",
        learning_rate=lr, per_device_train_batch_size=batch_size,
        per_device_eval_batch_size=batch_size, num_train_epochs=epochs,
        weight_decay=0.01, load_best_model_at_end=True,
        metric_for_best_model="eval_f1", greater_is_better=True,
        logging_steps=50, save_total_limit=1, report_to="none",
        fp16=torch.cuda.is_available(),
    )
    trainer = Trainer(model=model, args=args,
                      train_dataset=hf["train"], eval_dataset=hf["validation"],
                      processing_class=tok,
                      data_collator=DataCollatorWithPadding(tokenizer=tok),
                      compute_metrics=trainer_metrics)
    trainer.train()

    aug_probs_raw = []
    model.eval()
    for i in range(0, len(hf["test_aug"]), batch_size):
        batch = {k: v.to(DEVICE) for k, v in hf["test_aug"][i:i+batch_size].items()
                 if k in ["input_ids","attention_mask"]}
        with torch.no_grad():
            logits = model.to(DEVICE)(**batch).logits
            p = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()
        aug_probs_raw.extend(p.tolist())
    aug_probs = np.array(aug_probs_raw)

    clean_probs_raw = []
    for i in range(0, len(hf["test_clean"]), batch_size):
        batch = {k: v.to(DEVICE) for k, v in hf["test_clean"][i:i+batch_size].items()
                 if k in ["input_ids","attention_mask"]}
        with torch.no_grad():
            logits = model.to(DEVICE)(**batch).logits
            p = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()
        clean_probs_raw.extend(p.tolist())
    clean_probs = np.array(clean_probs_raw)

    val_probs_raw = []
    for i in range(0, len(hf["validation"]), batch_size):
        batch = {k: v.to(DEVICE) for k, v in hf["validation"][i:i+batch_size].items()
                 if k in ["input_ids","attention_mask"]}
        with torch.no_grad():
            logits = model.to(DEVICE)(**batch).logits
            p = torch.softmax(logits, dim=-1)[:,1].cpu().numpy()
        val_probs_raw.extend(p.tolist())
    val_probs = np.array(val_probs_raw)

    val_labels = np.array(val_df["label"].values)
    best_t, _ = find_best_thr_encoder(val_labels, val_probs)

    clean_m = compute_metrics_from_probs(test_clean_df["label"].values, clean_probs, best_t)
    aug_m   = compute_metrics_from_probs(test_aug_df["label"].values,   aug_probs,   best_t)

    trainer.save_model(output_dir)
    tok.save_pretrained(output_dir)

    result = {
        "model": model_name,
        "best_threshold": best_t,
        "clean_accuracy": clean_m["accuracy"], "clean_f1": clean_m["f1"],
        "clean_auc": clean_m["auc"],
        "attacked_accuracy": aug_m["accuracy"], "attacked_f1": aug_m["f1"],
        "attacked_auc": aug_m["auc"],
        "auc_drop": clean_m["auc"] - aug_m["auc"],
        "f1_drop":  clean_m["f1"]  - aug_m["f1"],
    }
    return result, aug_probs

print("train_encoder function defined.")

In [ ]:
# Запуск fine-tuning энкодеров
encoder_models = ["roberta-base", "answerdotai/ModernBERT-base"]
results = []; aug_probs_by_model = {}

for mname in encoder_models:
    safe = mname.replace("/","__")
    print(f"\n===== {mname} =====")
    res, aug_probs = train_encoder(
        model_name=mname, train_df=train_df, val_df=val_df,
        test_clean_df=test_clean_df, test_aug_df=test_aug_df,
        output_dir=f"/content/finetuned_models/{safe}",
        max_length=512, epochs=2, batch_size=8,
    )
    results.append(res); aug_probs_by_model[mname] = aug_probs

encoder_df = pd.DataFrame(results)
encoder_df.to_csv("/content/encoder_results.csv", index=False)
encoder_df

In [ ]:
# Анализ по уровням атаки для каждого энкодера
level_results = []
for row in results:
    mname    = row["model"]
    threshold = row["best_threshold"]
    probs     = aug_probs_by_model[mname]
    ldf = evaluate_by_distortion_level(test_aug_df, probs, threshold)
    ldf["model"] = mname
    level_results.append(ldf)

level_df = pd.concat(level_results, ignore_index=True)
level_df.to_csv("/content/encoder_by_level.csv", index=False)
print(level_df[["model","distortion_level","f1","auc","n"]])

In [ ]:
def make_imgtb_compatible(src_dir, dst_dir):
    os.makedirs(dst_dir, exist_ok=True)
    model = AutoModelForSequenceClassification.from_pretrained(src_dir)
    tok   = AutoTokenizer.from_pretrained(src_dir)
    if hasattr(model, "classifier") and hasattr(model.classifier, "out_proj"):
        layer = model.classifier.out_proj
    elif hasattr(model, "classifier") and hasattr(model.classifier, "weight"):
        layer = model.classifier
    else:
        raise ValueError("Не удалось найти classification head.")
    with torch.no_grad():
        layer.weight.data = layer.weight.data[[1,0],:].clone()
        layer.bias.data   = layer.bias.data[[1,0]].clone()
    model.config.id2label = {0:"MACHINE", 1:"HUMAN"}
    model.config.label2id = {"MACHINE":0, "HUMAN":1}
    model.save_pretrained(dst_dir); tok.save_pretrained(dst_dir)
    print("Saved IMGTB-compatible model:", dst_dir)

for mname in encoder_models:
    safe = mname.replace("/","__")
    make_imgtb_compatible(
        f"/content/finetuned_models/{safe}",
        f"/content/finetuned_models_imgtb/{safe}",
    )

In [ ]:
print("Запуск IMGTB для fine-tuned моделей:")
for mname in encoder_models:
    safe = mname.replace("/","__")
    for split, csv in [("clean", "clean_test"), ("attacked", "attacked_test")]:
        print(f"!python benchmark.py \\")
        print(f"  --dataset /content/{csv}.csv csv default text label 0 \\")
        print(f"  --methods /content/finetuned_models_imgtb/{safe} \\")
        print(f"  --name {split}_finetuned_{safe}")
        print()

## 14. Дообучение Gemma

Дообучаем Gemma-3-1B-IT (и опционально Gemma-4-E2B-IT) как модель переписывания текстов
с управляемыми уровнями искажения (weak / medium / strong) через LoRA.

In [ ]:
GEMMA_3_1B    = "google/gemma-3-1b-it"
GEMMA_4_E2B   = "google/gemma-4-E2B-it"
GEMMA_LORA_PATH_3B = "gemma_rewriter_lora"
GEMMA_LORA_PATH_4B = "gemma_4_rewriter_lora"

HF_TOKEN = os.getenv("HF_TOKEN")

GEMMA_SYSTEM_PROMPT = (
    "You are a text corruption and rewriting assistant. "
    "Given an original text and a rewrite level, produce a modified version of the text. "
    "Your goal is to introduce realistic local distortions while preserving the original meaning. "
    "Possible distortions include typos, swapped letters, missing letters, extra letters, "
    "spacing mistakes, punctuation mistakes, grammar mistakes, visually similar character substitutions, "
    "and small word-form distortions. "
    "Do not add new facts. Do not summarize. Do not simplify. "
    "Do not fully paraphrase the text. "
    "Keep the wording and sentence structure as close to the original as possible. "
    "The output must not be identical to the original text. "
    "Return only the rewritten text."
)

def build_user_prompt(example):
    level = example.get("effective_level", example.get("distortion_level", "medium"))
    req = {
        "weak":   "Introduce a few minor visible distortions.",
        "medium": "Introduce several noticeable visible distortions.",
        "strong": "Introduce many visible distortions, but keep the text recognizable. "
                  "Corrupt multiple words in most sentences.",
    }[level]
    return (f"Rewrite level: {level}\n{req}\n\n"
            f"Original text:\n{example['text']}\n\nRewritten text:")

print("Gemma config defined.")

In [ ]:
# Загрузка датасета для Gemma
gemma_dataset = load_from_disk("data/gemma_rewrite_dataset_effective")
train_rw = gemma_dataset["train"].shuffle(seed=42).select(range(min(5000, len(gemma_dataset["train"]))))
val_rw   = gemma_dataset["validation"].shuffle(seed=42).select(range(min(400, len(gemma_dataset["validation"]))))
test_rw  = gemma_dataset["test"]
print(f"train_rw: {len(train_rw)}, val_rw: {len(val_rw)}, test_rw: {len(test_rw)}")

In [ ]:
# Инициализация Gemma-3-1B-IT с LoRA
gemma_tokenizer = AutoTokenizer.from_pretrained(GEMMA_3_1B, token=HF_TOKEN)
if gemma_tokenizer.pad_token is None:
    gemma_tokenizer.pad_token = gemma_tokenizer.eos_token

gemma_model = AutoModelForCausalLM.from_pretrained(
    GEMMA_3_1B, token=HF_TOKEN, device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)
gemma_model.config.use_cache = False
gemma_model.gradient_checkpointing_enable()

peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)
gemma_model = get_peft_model(gemma_model, peft_config)
gemma_model.print_trainable_parameters()

In [ ]:
# Токенизация датасета для обучения
GEMMA_MAX_LEN = 768

def build_chat_prompt_gemma(example):
    messages = [
        {"role": "system",  "content": GEMMA_SYSTEM_PROMPT},
        {"role": "user",    "content": build_user_prompt(example)},
    ]
    return gemma_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def preprocess_gemma(example):
    prompt = build_chat_prompt_gemma(example)
    completion = example["broken_text"].strip() + gemma_tokenizer.eos_token

    prompt_ids     = gemma_tokenizer(prompt, add_special_tokens=False)["input_ids"]
    completion_ids = gemma_tokenizer(completion, add_special_tokens=False)["input_ids"]

    max_comp_len = min(len(completion_ids), 384)
    completion_ids = completion_ids[:max_comp_len]
    max_prompt_len = GEMMA_MAX_LEN - len(completion_ids)
    if max_prompt_len <= 0:
        completion_ids = completion_ids[:GEMMA_MAX_LEN]; prompt_ids = []
    else:
        prompt_ids = prompt_ids[-max_prompt_len:]

    input_ids      = prompt_ids + completion_ids
    labels         = [-100] * len(prompt_ids) + completion_ids
    attention_mask = [1] * len(input_ids)

    return {"input_ids": input_ids, "attention_mask": attention_mask, "labels": labels}

train_tok_rw = train_rw.map(preprocess_gemma, remove_columns=train_rw.column_names)
val_tok_rw   = val_rw.map(preprocess_gemma,   remove_columns=val_rw.column_names)
print("Tokenized:", train_tok_rw[0].keys())

In [ ]:
# Обучение Gemma-3-1B-IT
data_collator_gemma = DataCollatorForSeq2Seq(
    tokenizer=gemma_tokenizer, model=gemma_model,
    padding=True, label_pad_token_id=-100,
)

gemma_training_args = TrainingArguments(
    output_dir=GEMMA_LORA_PATH_3B,
    per_device_train_batch_size=2, per_device_eval_batch_size=2,
    gradient_accumulation_steps=16, num_train_epochs=3,
    learning_rate=1e-4, lr_scheduler_type="cosine", warmup_steps=100,
    logging_steps=20, eval_strategy="no", save_strategy="epoch",
    save_total_limit=2,
    bf16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8,
    fp16=torch.cuda.is_available() and torch.cuda.get_device_capability()[0] < 8,
    report_to="none",
)

gemma_trainer = Trainer(
    model=gemma_model, args=gemma_training_args,
    train_dataset=train_tok_rw, eval_dataset=val_tok_rw,
    data_collator=data_collator_gemma,
)
gemma_trainer.train()
gemma_trainer.save_model(GEMMA_LORA_PATH_3B)
gemma_tokenizer.save_pretrained(GEMMA_LORA_PATH_3B)
print("Gemma-3-1B LoRA saved:", GEMMA_LORA_PATH_3B)
# !zip -r gemma_rewriter_lora.zip gemma_rewriter_lora/

In [ ]:
# Загрузка обученной Gemma для инференса
from peft import PeftModel

def load_gemma_for_inference(base_model_name, lora_path):
    tok = AutoTokenizer.from_pretrained(lora_path)
    base = AutoModelForCausalLM.from_pretrained(
        base_model_name, device_map="auto",
        torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    )
    model = PeftModel.from_pretrained(base, lora_path)
    model.eval()
    return tok, model

TARGET_BANDS = {
    "weak":   (0.005, 0.025),
    "medium": (0.020, 0.050),
    "strong": (0.040, 0.090),
}

def clean_gemma_output(s):
    s = s.strip()
    if s.startswith("model"): s = s[5:].strip()
    if "Rewritten text:" in s: s = s.split("Rewritten text:", 1)[-1].strip()
    return s.strip()

def choose_best_candidate(src, candidates, level):
    lo, hi = TARGET_BANDS[level]
    src = src.strip(); src_len = max(len(src), 1)
    scored = []
    for c in candidates:
        c = clean_gemma_output(c)
        if not c: continue
        d_norm = Levenshtein.distance(src, c) / src_len
        penalty = 1000.0 if c == src else 0.0
        if d_norm < lo: penalty += (lo - d_norm) * 1000
        elif d_norm > hi: penalty += (d_norm - hi) * 1000
        scored.append((penalty, abs(((lo + hi) / 2) - d_norm), d_norm, c))
    if not scored: return candidates[0].strip() if candidates else ""
    scored.sort(key=lambda x: (x[0], x[1]))
    return scored[0][3]

def generate_rewrite_gemma(text, tokenizer, model, level="medium", max_new_tokens=256):
    req = {"weak": "Introduce a few minor visible distortions.",
           "medium": "Introduce several noticeable visible distortions.",
           "strong": "Introduce many visible distortions, but keep the text recognizable. Corrupt multiple words."}[level]
    messages = [{"role": "system", "content": GEMMA_SYSTEM_PROMPT},
                {"role": "user", "content": f"Rewrite level: {level}\n{req}\n\nOriginal text:\n{text}\n\nRewritten text:"}]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    num_seq = {"weak": 4, "medium": 6, "strong": 8}[level]
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=True,
                                  temperature=0.95, top_p=0.95, repetition_penalty=1.03,
                                  num_return_sequences=num_seq,
                                  pad_token_id=tokenizer.pad_token_id,
                                  eos_token_id=tokenizer.eos_token_id)
    candidates = []
    for out in outputs:
        full = tokenizer.decode(out, skip_special_tokens=True)
        cand = full.split("Rewritten text:", 1)[-1].strip() if "Rewritten text:" in full else full.strip()
        candidates.append(clean_gemma_output(cand))
    return choose_best_candidate(text, list(dict.fromkeys(candidates)), level)

# Загрузка и тест
# gemma_tok_inf, gemma_model_inf = load_gemma_for_inference(GEMMA_3_1B, GEMMA_LORA_PATH_3B)
# sample = test_rw[0]["text"]
# for level in ["weak", "medium", "strong"]:
#     out = generate_rewrite_gemma(sample, gemma_tok_inf, gemma_model_inf, level=level)
#     d = Levenshtein.distance(sample, out) / max(len(sample), 1)
#     print(f"{level}: dist_norm={d:.4f}  |  {out[:80]}...")
print("Gemma inference functions defined.")

## 15. Проверка декодеров

Тестируем, могут ли LLM-декодеры (Gemma-2-2B-IT, Qwen2.5-3B-Instruct)
определять AI-текст через zero-shot промптинг.

In [ ]:
def make_detector_prompt(text):
    return (
        "You are an AI text detector.\n\n"
        "Classify the following text as either HUMAN or MACHINE.\n\n"
        "Return only one word: HUMAN or MACHINE.\n\n"
        f"Text:\n{text}\n\nAnswer:"
    )

def load_decoder_model(model_name):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type="nf4")
    tok   = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant,
                                                  device_map="auto")
    return tok, model

def predict_decoder(text, tokenizer, model, max_new_tokens=5):
    prompt = make_detector_prompt(text)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                       max_length=2048).to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                              pad_token_id=tokenizer.eos_token_id)
    answer = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:],
                               skip_special_tokens=True).strip().upper()
    if "MACHINE" in answer or "AI" in answer: return 1, answer
    elif "HUMAN" in answer: return 0, answer
    return -1, answer

def evaluate_decoder_model(df, model_name, split_name):
    from tqdm.auto import tqdm
    tok, model = load_decoder_model(model_name)
    preds, raw_answers = [], []
    for text in tqdm(df["text"], desc=f"{model_name}/{split_name}"):
        pred, raw = predict_decoder(text, tok, model)
        preds.append(pred); raw_answers.append(raw)
    res_df = df.copy()
    res_df["pred"] = preds; res_df["raw_answer"] = raw_answers
    valid = res_df[res_df["pred"] != -1].copy()
    y_true, y_pred = valid["label"].values, valid["pred"].values
    prec, rec, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary",
                                                        zero_division=0)
    return {"model": model_name, "split": split_name,
            "n_total": len(res_df), "n_valid": len(valid),
            "invalid_rate": 1 - len(valid)/len(res_df),
            "accuracy": accuracy_score(y_true, y_pred),
            "f1": f1, "precision": prec, "recall": rec}, res_df

clean_df = dataset["test_clean_eval"].to_pandas()[["text","label"]]
clean_df["label"] = 1 - clean_df["label"]
aug_df_dec = dataset["test_aug_eval"].to_pandas()[["broken_text","label"]].rename(
    columns={"broken_text":"text"})
aug_df_dec["label"] = 1 - aug_df_dec["label"]

clean_sample   = clean_df.sample(200, random_state=42).reset_index(drop=True)
attacked_sample = aug_df_dec.sample(200, random_state=42).reset_index(drop=True)

print("Decoder test samples:", len(clean_sample), len(attacked_sample))

In [ ]:
# Запуск оценки декодеров
DECODER_MODELS = ["google/gemma-2-2b-it", "Qwen/Qwen2.5-3B-Instruct"]

decoder_results = []; decoder_outputs = {}

for model_name in DECODER_MODELS:
    for split_name, df in [("clean", clean_sample), ("attacked", attacked_sample)]:
        metrics, outputs = evaluate_decoder_model(df, model_name, split_name)
        decoder_results.append(metrics)
        decoder_outputs[(model_name, split_name)] = outputs
        outputs.to_csv(f"/content/decoder_{model_name.replace('/','__')}_{split_name}.csv",
                       index=False)

decoder_df = pd.DataFrame(decoder_results)
decoder_df.to_csv("/content/decoder_results.csv", index=False)
decoder_df

## 16. Комбинирование Gemma и правил

Сравниваем четыре стратегии состязательных атак:
1. **rules-only** — только правила из фреймворка
2. **gemma-only** — только Gemma переписывает текст
3. **rules → gemma** — сначала правила, потом Gemma
4. **gemma → rules** — сначала Gemma, потом правила

In [ ]:
from sentence_transformers import SentenceTransformer, util as st_util
from rules.text_utils import normalize_spacing

GEMMA_MODEL_NAME = "google/gemma-3-1b-it"
RANDOM_SEED = 42

def load_gemma_attacker(model_name=GEMMA_MODEL_NAME):
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16,
                                bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True)
    tok = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(model_name, quantization_config=quant,
                                                  device_map={"":0}, torch_dtype=torch.float16,
                                                  trust_remote_code=True)
    model.eval(); return tok, model

MAX_INPUT_CHARS = 4000

def gemma_rewrite(text, gemma_tok, gemma_mdl, max_new_tokens=256, temperature=0.95):
    text = text[:MAX_INPUT_CHARS]
    prompt = (
        "\nYou are rewriting text to evade AI text detectors.\n"
        "Rewrite the text substantially while preserving meaning, facts, and length.\n"
        "Sound natural. Vary wording. Avoid formal AI-style tone.\n"
        "Return ONLY the rewritten text.\n"
        f"\nTEXT:\n{text}\n\nREWRITTEN:\n"
    )
    inputs = gemma_tok(prompt, return_tensors="pt", truncation=True,
                       max_length=2048).to(gemma_mdl.device)
    input_len = inputs["input_ids"].shape[-1]
    with torch.no_grad():
        outputs = gemma_mdl.generate(**inputs, max_new_tokens=max_new_tokens,
                                      do_sample=True, temperature=temperature,
                                      top_p=0.92, top_k=50, repetition_penalty=1.1,
                                      no_repeat_ngram_size=4,
                                      pad_token_id=gemma_tok.eos_token_id)
    generated = gemma_tok.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
    for pfx in ["Rewritten text:", "Here is", "Sure,"]:
        if generated.startswith(pfx): generated = generated[len(pfx):].strip()
    return normalize_spacing(generated)

ppl_tok_hybrid, ppl_mdl_hybrid = None, None

def get_ppl_model():
    global ppl_tok_hybrid, ppl_mdl_hybrid
    if ppl_mdl_hybrid is None:
        ppl_tok_hybrid = AutoTokenizer.from_pretrained("gpt2")
        ppl_mdl_hybrid = AutoModelForCausalLM.from_pretrained("gpt2").to(DEVICE)
        ppl_mdl_hybrid.eval()
        if ppl_tok_hybrid.pad_token is None: ppl_tok_hybrid.pad_token = ppl_tok_hybrid.eos_token
    return ppl_tok_hybrid, ppl_mdl_hybrid

def compute_ppl_text(text, max_length=512):
    tok, mdl = get_ppl_model()
    enc = tok(text, return_tensors="pt", truncation=True, max_length=max_length).to(DEVICE)
    with torch.no_grad():
        loss = mdl(**enc, labels=enc["input_ids"]).loss.item()
    return float(math.exp(min(loss, 20)))

sem_model_hybrid = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=DEVICE)

def semantic_sim(a, b):
    emb = sem_model_hybrid.encode([a, b], convert_to_tensor=True, normalize_embeddings=True)
    return float(st_util.cos_sim(emb[0], emb[1]).item())

def edit_ratio(a, b):
    return Levenshtein.distance(a, b) / max(len(a), 1)

def eval_quality(orig, attacked):
    ppl_o = compute_ppl_text(orig)
    ppl_a = compute_ppl_text(attacked)
    read_o, _ = human_readability_score(orig)
    read_a, _ = human_readability_score(attacked)
    return {"semantic_similarity": semantic_sim(orig, attacked),
            "edit_distance_ratio": edit_ratio(orig, attacked),
            "ppl_original": ppl_o, "ppl_attacked": ppl_a,
            "ppl_ratio": ppl_a / max(ppl_o, 1e-8),
            "readability_delta": read_a - read_o, "readability_attacked": read_a,
            "length_ratio": len(attacked) / max(len(orig), 1)}

print("Hybrid attack functions defined.")

In [ ]:
# Четыре стратегии атаки
def rules_attack(text, seed=0, tries=16, strength=0.5, k_max=3):
    res = fw.generate_best_candidate(text, seed=seed, tries=tries, strength=strength,
                                     k_max=k_max, target_drop=0.0)
    return res["text"], [x["rule"] for x in res["log"]]

def run_hybrid_experiment(df, gemma_tok, gemma_mdl, seed=42, tries=16, strength=0.5, k_max=3):
    from tqdm.auto import tqdm
    rows = []
    for idx, row in tqdm(df.iterrows(), total=len(df)):
        text = row["text"]
        base = fw.measure(text)

        strategies = {}
        # 1. Rules only
        t1, rules1 = rules_attack(text, seed=seed+idx, tries=tries, strength=strength, k_max=k_max)
        strategies["rules_only"] = (t1, rules1)
        # 2. Gemma only
        t2 = gemma_rewrite(text, gemma_tok, gemma_mdl)
        strategies["gemma_only"] = (t2, [])
        # 3. Rules → Gemma
        t3 = gemma_rewrite(t1, gemma_tok, gemma_mdl)
        strategies["rules_then_gemma"] = (t3, rules1)
        # 4. Gemma → Rules
        t4, rules4 = rules_attack(t2, seed=seed+idx+20000, tries=tries, strength=strength, k_max=k_max)
        strategies["gemma_then_rules"] = (t4, rules4)

        for strategy, (attacked, rules) in strategies.items():
            cand = fw.measure(attacked)
            q = eval_quality(text, attacked)
            rows.append({"id": idx, "strategy": strategy,
                         "original_text": text, "attacked_text": attacked,
                         "rules": str(rules),
                         "base_radar_ai": base["radar_ai"],
                         "attacked_radar_ai": cand["radar_ai"],
                         "radar_score_drop": base["radar_ai"] - cand["radar_ai"],
                         "changed": int(text != attacked), **q})
    return pd.DataFrame(rows)


gemma_tok_att, gemma_mdl_att = load_gemma_attacker()
machine_sample = machine_texts[:30]
sample_machine_df = pd.DataFrame({"text": machine_sample, "label": [1]*len(machine_sample)})
hybrid_df = run_hybrid_experiment(sample_machine_df, gemma_tok_att, gemma_mdl_att, seed=RANDOM_SEED)
hybrid_df.to_csv("/content/hybrid_results.csv", index=False)

In [ ]:
# Анализ результатов гибридных стратегий
hybrid_df = pd.read_csv("/content/hybrid_results.csv")

# Сводная таблица
summary = hybrid_df.groupby("strategy").agg(
    n=("id","count"), changed_rate=("changed","mean"),
    radar_drop_mean=("radar_score_drop","mean"),
    radar_drop_median=("radar_score_drop","median"),
    semantic_sim_mean=("semantic_similarity","mean"),
    edit_ratio_mean=("edit_distance_ratio","mean"),
    ppl_ratio_mean=("ppl_ratio","mean"),
   readability_delta_mean=("readability_delta","mean"),
).reset_index()
print(summary)

print("Пример результатов:")
summary_example = pd.DataFrame([
    {"strategy":"rules_only",      "radar_drop":0.073, "semantic_sim":0.955, "ppl_ratio":2.06, "edit_ratio":0.031},
    {"strategy":"gemma_only",      "radar_drop":-0.269,"semantic_sim":0.819, "ppl_ratio":2.29, "edit_ratio":0.947},
    {"strategy":"rules_then_gemma","radar_drop":-0.273,"semantic_sim":0.808, "ppl_ratio":2.32, "edit_ratio":0.946},
    {"strategy":"gemma_then_rules","radar_drop":-0.213,"semantic_sim":0.772, "ppl_ratio":3.91, "edit_ratio":1.059},
])
print(summary_example)

## 17. Анализ правил: какие стратегии улучшают/ухудшают детекцию

Тестируем каждое правило отдельно,
анализируем влияние силы атаки (strength) и числа комбинируемых правил (k_max).

In [ ]:
def evaluate_single_rules(texts, fw, n_runs=3, strength=0.5, seed=42):
    rows = []
    for i, text in enumerate(tqdm(texts)):
        base = fw.measure(text)
        for rule_name in fw.rulebook.keys():
            best_row = None; best_drop = -1e9
            for r in range(n_runs):
                run_seed = seed + i * 10_000 + r
                try:
                    cand_text, meta = fw.apply_rule(text, rule_name, seed=run_seed, strength=strength)
                    cand = fw.measure(cand_text)
                    drop = base["radar_ai"] - cand["radar_ai"]
                    if drop > best_drop:
                        best_drop = drop
                        best_row = {"id": i, "rule": rule_name,
                                    "bucket": fw.rulebook[rule_name].bucket,
                                    "base_radar_ai": base["radar_ai"],
                                    "attacked_radar_ai": cand["radar_ai"],
                                    "radar_score_drop": drop,
                                    "base_ppl": base["ppl"], "attacked_ppl": cand["ppl"],
                                    "changed": int(text != cand_text),
                                    "original_text": text, "attacked_text": cand_text}
                except: continue
            if best_row: rows.append(best_row)
    return pd.DataFrame(rows)

sample_texts = machine_texts[:50]
single_rule_df = evaluate_single_rules(sample_texts, fw, n_runs=3, strength=0.5)
print("Single rule ablation done:", single_rule_df.shape)

In [ ]:
quality_rows = []
for _, row in tqdm(single_rule_df.iterrows(), total=len(single_rule_df)):
    q = eval_quality(row["original_text"], row["attacked_text"])
    quality_rows.append(q)
single_rule_df = pd.concat([single_rule_df.reset_index(drop=True),
                              pd.DataFrame(quality_rows)], axis=1)
single_rule_df.to_csv("/content/single_rule_ablation.csv", index=False)

# Сводка по правилам
rule_summary = single_rule_df.groupby(["rule","bucket"]).agg(
    n=("id","count"), changed_rate=("changed","mean"),
    radar_drop_mean=("radar_score_drop","mean"),
    radar_drop_median=("radar_score_drop","median"),
    semantic_sim_mean=("semantic_similarity","mean"),
    edit_ratio_mean=("edit_distance_ratio","mean"),
    ppl_ratio_mean=("ppl_ratio","mean"),
    readability_delta_mean=("readability_delta","mean"),
).reset_index().sort_values("radar_drop_mean", ascending=False)
rule_summary.to_csv("/content/rule_summary.csv", index=False)
print(rule_summary[["rule","bucket","radar_drop_mean","semantic_sim_mean","ppl_ratio_mean"]])

In [ ]:
# Сводка по бакетам
bucket_summary = single_rule_df.groupby("bucket").agg(
    n=("id","count"),
    radar_drop_mean=("radar_score_drop","mean"),
    semantic_sim_mean=("semantic_similarity","mean"),
    ppl_ratio_mean=("ppl_ratio","mean"),
).reset_index().sort_values("radar_drop_mean", ascending=False)
print(bucket_summary)

In [ ]:
# Влияние силы атаки
strength_map = {"weak": 0.25, "medium": 0.50, "strong": 0.85}
strength_rows = []

for strength_name, strength in strength_map.items():
    df_s = evaluate_single_rules(sample_texts, fw, n_runs=2, strength=strength)
    df_s["strength_name"] = strength_name; df_s["strength"] = strength
    strength_rows.append(df_s)

strength_df = pd.concat(strength_rows, ignore_index=True)

strength_summary = strength_df.groupby(["strength_name","strength"]).agg(
    radar_drop_mean=("radar_score_drop","mean"),
    changed_rate=("changed","mean"),
).reset_index()

print("Влияние силы атаки:")
print(strength_summary)

In [ ]:
# Влияние числа комбинируемых правил
combo_rows = []
for k_max in [1, 2, 3, 4]:
    for strength_name, strength in strength_map.items():
        for i, text in enumerate(tqdm(sample_texts, desc=f"k={k_max},{strength_name}")):
            res = fw.generate_best_candidate(text, tries=16, strength=strength,
                                              seed=42+i, k_max=k_max)
            base = res["base"]; cand = res["cand"]
            q = eval_quality(text, res["text"])
            combo_rows.append({"id":i, "k_max":k_max, "strength_name":strength_name,
                                "strength":strength, "combo":str(res["combo"]),
                                "base_radar_ai":base["radar_ai"],
                                "attacked_radar_ai":cand["radar_ai"],
                                "radar_score_drop":base["radar_ai"]-cand["radar_ai"],
                                "changed":int(text!=res["text"]), **q})

combo_df = pd.DataFrame(combo_rows)
combo_summary = combo_df.groupby(["k_max","strength_name"]).agg(
    radar_drop_mean=("radar_score_drop","mean"),
    semantic_sim_mean=("semantic_similarity","mean"),
    ppl_ratio_mean=("ppl_ratio","mean"),
    changed_rate=("changed","mean"),
).reset_index()

print("Влияние k_max:")
print(combo_summary.pivot(index="strength_name", columns="k_max", values="radar_drop_mean").round(4))

In [ ]:
# Статистическая значимость (тест Уилкоксона)
print("Тест Уилкоксона: правила")
rule_stats = []
for rule_name in single_rule_df["rule"].unique():
    sub = single_rule_df[single_rule_df["rule"] == rule_name]
    try:
        stat, p = wilcoxon(sub["base_radar_ai"], sub["attacked_radar_ai"])
        rule_stats.append({"rule": rule_name, "stat": stat, "pvalue": p})
    except: pass
print(pd.DataFrame(rule_stats).sort_values("pvalue").to_string(index=False))

print("\nТест Уилкоксона: бакеты")
for bucket in single_rule_df["bucket"].unique():
    sub = single_rule_df[single_rule_df["bucket"] == bucket]
    stat, p = wilcoxon(sub["base_radar_ai"], sub["attacked_radar_ai"])
    print(f"  {bucket}: stat={stat:.1f}, p={p:.4f}")

print("\nТест Уилкоксона: комбо (k=3, medium)")
sub = combo_df[(combo_df["k_max"]==3) & (combo_df["strength_name"]=="medium")]
stat, p = wilcoxon(sub["base_radar_ai"], sub["attacked_radar_ai"])
print(f"  k=3, medium: stat={stat:.1f}, p={p:.4f}")